[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Performance &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
import timeit
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


**1.** Time `np.nanmean(readings[2])` against a loop that keeps Svalbard's readings that are not `None` in `readings_list[2]` and divides their sum by their count, check that the two means agree to two places, and print whether the loop took more than twice as long.

In [2]:
def best(function, number=20):
    return min(timeit.repeat(function, number=number, repeat=5))


def svalbard_mean_with_a_loop():
    present = [celsius for celsius in readings_list[2] if celsius is not None]
    return sum(present) / len(present)


print(round(svalbard_mean_with_a_loop(), 2), np.nanmean(readings[2]).round(2))
print("the loop took more than twice as long:",
      best(svalbard_mean_with_a_loop) > 2 * best(lambda: np.nanmean(readings[2])))

-4.48 -4.48
the loop took more than twice as long: True


Python's `sum` is written in C, and `np.nanmean` finds the `nan` values before it adds, so the two are closer than the worked examples' loops, and the cell asks for a smaller margin.

**2.** Make Bergen's readings as `int16` tenths of a degree, `np.round(bergen * 10).astype(np.int16)`, and print its `itemsize` and `nbytes` beside those of the `float64` row.

In [3]:
bergen = readings[0]
tenths = np.round(bergen * 10).astype(np.int16)

print(tenths.itemsize, tenths.nbytes)
print(bergen.itemsize, bergen.nbytes)

2 17520
8 70080


Two bytes a value instead of eight, a quarter of the memory, and every reading still exact to the tenth of a degree it was recorded to.

**3.** Count Oslo's hours above 15 degrees with a loop over `readings_list[1]` and with a mask, check that the counts agree, and print whether the loop took more than five times as long.

In [4]:
def warm_with_a_loop():
    return sum(1 for celsius in readings_list[1] if celsius > 15)


def warm_with_a_mask():
    return (readings[1] > 15).sum()


print(warm_with_a_loop(), warm_with_a_mask())
print("the loop took more than five times as long:", best(warm_with_a_loop) > 5 * best(warm_with_a_mask))

1043 1043
the loop took more than five times as long: True


The same pattern as the frost count in the worked examples, on another station.

**4.** Build Tromso's readings in Fahrenheit by appending to a list in a loop and converting once with `np.array`, and check with `np.array_equal` that the result equals `readings[3] * 9 / 5 + 32`.

In [5]:
tromso_fahrenheit = []
for celsius in readings_list[3]:
    tromso_fahrenheit.append(celsius * 9 / 5 + 32)

print(np.array_equal(np.array(tromso_fahrenheit), readings[3] * 9 / 5 + 32))

True


The loop and the array do the same three operations in the same order, so every value matches exactly.

**5.** Time `np.vectorize(math.sqrt)` against `np.sqrt` on the absolute values of Bergen's readings, and print whether `np.vectorize` took more than ten times as long.

In [6]:
distances = np.abs(readings[0])
vectorized_sqrt = np.vectorize(math.sqrt)

print(np.allclose(vectorized_sqrt(distances), np.sqrt(distances)))
print("np.vectorize took more than ten times as long:",
      best(lambda: vectorized_sqrt(distances)) > 10 * best(lambda: np.sqrt(distances)))

True
np.vectorize took more than ten times as long: True


`math.sqrt` handles one Python float at a time, and `np.vectorize` calls it 8,760 times.

---

&#8592; **Back to:** [Performance](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/14-performance.ipynb)  &nbsp;&middot;&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
